In [ ]:
import torch
import torch.nn as nn

class VectorField(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2)
        )
    def forward(self, t, y):
        return self.net(y)

def rk4_step(f, t, y, dt):
    """Custom RK4 Integrator built from scratch[cite: 5]."""
    k1 = f(t, y)
    k2 = f(t + dt/2, y + dt/2 * k1)
    k3 = f(t + dt/2, y + dt/2 * k2)
    k4 = f(t + dt, y + dt * k3)
    return y + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

def train_neural_ode(t_data, y_data, epochs=1000):
    """Trains a Neural ODE recognizing trajectory without providing physics equations[cite: 5]."""
    print("--- Training Neural ODE (Relies on Trajectory Data, 0 Equations) ---")
    f_net = VectorField()
    optimizer = torch.optim.Adam(f_net.parameters(), lr=2e-3)
    
    dt = t_data[1] - t_data[0]
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        y_pred = [y_data[0]] 
        for i in range(len(t_data) - 1):
            y_next = rk4_step(f_net, t_data[i], y_pred[-1], dt)
            y_pred.append(y_next)
            
        y_pred_tensor = torch.stack(y_pred)
        
        loss = torch.mean((y_pred_tensor - y_data)**2)
        loss.backward()
        optimizer.step()
        
    return f_net